# HMS Harmful Brain Activity Classification — Keras Inference

Submission notebook for the KerasCV EfficientNetV2-B2 spectrogram model.

This model uses **Kaggle-provided spectrograms** (not EEG-derived).
- Preset: `efficientnetv2_b2_imagenet`
- Input: [400, 300, 3] log-normalized spectrogram
- Single fold (fold 0)
- Saved as `.keras` file

In [1]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

from pathlib import Path

import keras_cv
import keras
import tensorflow as tf

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import joblib

# Enable mixed precision + XLA if GPU available
gpus = tf.config.list_physical_devices('GPU')
print(f"Num GPUs: {len(gpus)}")
if len(gpus) > 0:
    print(f"GPU: {gpus[0]}")
    keras.mixed_precision.set_global_policy("mixed_float16")
    tf.config.optimizer.set_jit(True)

print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")


2026-03-05 03:14:30.380052: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772680470.582832      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772680470.638393      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772680471.099038      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772680471.099085      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772680471.099088      23 computation_placer.cc:177] computation placer alr

Num GPUs: 1
GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
TensorFlow: 2.19.0
Keras: 3.10.0


## Configuration

In [2]:
class CFG:
    seed = 42
    preset = "efficientnetv2_b2_imagenet"
    image_size = [400, 300]
    batch_size = 32
    num_classes = 6
    class_names = ['Seizure', 'LPD', 'GPD', 'LRDA', 'GRDA', 'Other']
    label2name = dict(enumerate(class_names))
    name2label = {v: k for k, v in label2name.items()}


print(f"Preset: {CFG.preset}")
print(f"Image size: {CFG.image_size}")


Preset: efficientnetv2_b2_imagenet
Image size: [400, 300]


## Paths

In [3]:
# Competition data
BASE_PATH = Path("/kaggle/input/competitions/hms-harmful-brain-activity-classification")

# ============================================================
# ADJUST THIS: point to your uploaded Keras model
# ============================================================
MODEL_PATH = Path("/kaggle/input/models/andrewpearlee/baseline/keras/default/1/best_model.keras")

# Working directory for processed spectrograms
SPEC_DIR = Path("/kaggle/temp/spectrograms")
SPEC_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model: {MODEL_PATH}")
print(f"Model exists: {MODEL_PATH.exists()}")


Model: /kaggle/input/models/andrewpearlee/baseline/keras/default/1/best_model.keras
Model exists: True


## Load Test Metadata

In [4]:
test_df = pd.read_csv(BASE_PATH / "test.csv")
test_df["spec_path"] = test_df["spectrogram_id"].apply(
    lambda x: str(BASE_PATH / "test_spectrograms" / f"{x}.parquet")
)
test_df["spec2_path"] = test_df["spectrogram_id"].apply(
    lambda x: str(SPEC_DIR / f"{x}.npy")
)
print(f"Test samples: {len(test_df)}")
print(test_df.head())


Test samples: 1
   spectrogram_id      eeg_id  patient_id  \
0          853520  3911565283        6885   

                                           spec_path  \
0  /kaggle/input/competitions/hms-harmful-brain-a...   

                             spec2_path  
0  /kaggle/temp/spectrograms/853520.npy  


## Convert Spectrograms (parquet → npy)

In [5]:
def process_spec(spec_id):
    """Convert one spectrogram from parquet to npy."""
    output_path = SPEC_DIR / f"{spec_id}.npy"
    if output_path.exists():
        return
    spec_path = BASE_PATH / "test_spectrograms" / f"{spec_id}.parquet"
    spec = pd.read_parquet(str(spec_path))
    spec = spec.fillna(0).values[:, 1:].T  # drop time col, transpose to (Freq, Time)
    spec = spec.astype("float32")
    np.save(str(output_path), spec)


spec_ids = test_df["spectrogram_id"].unique()
print(f"Processing {len(spec_ids)} test spectrograms...")
_ = joblib.Parallel(n_jobs=-1, backend="loky")(
    joblib.delayed(process_spec)(sid)
    for sid in tqdm(spec_ids, total=len(spec_ids))
)
print("Done.")


Processing 1 test spectrograms...


  0%|          | 0/1 [00:00<?, ?it/s]

Done.


## DataLoader

In [6]:
def build_decoder(target_size=CFG.image_size, dtype=32):
    """Decode .npy spectrogram for inference (no labels, no offset)."""
    def decode_signal(path):
        file_bytes = tf.io.read_file(path)
        sig = tf.io.decode_raw(file_bytes, tf.float32)
        sig = sig[1024 // dtype:]  # remove npy header
        sig = tf.reshape(sig, [400, -1])

        # Log spectrogram
        sig = tf.clip_by_value(sig, tf.math.exp(-4.0), tf.math.exp(8.0))
        sig = tf.math.log(sig)

        # Normalize
        sig -= tf.math.reduce_mean(sig)
        sig /= tf.math.reduce_std(sig) + 1e-6

        # Mono -> 3 channels (for ImageNet pretrained weights)
        sig = tf.tile(sig[..., None], [1, 1, 3])
        return sig

    return decode_signal


def build_test_dataset(paths, batch_size=32):
    AUTO = tf.data.experimental.AUTOTUNE
    decode_fn = build_decoder()
    ds = tf.data.Dataset.from_tensor_slices(paths)
    ds = ds.map(decode_fn, num_parallel_calls=AUTO)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(AUTO)
    return ds


test_paths = test_df.spec2_path.values
test_ds = build_test_dataset(test_paths, batch_size=min(CFG.batch_size, len(test_df)))
print(f"Test batches: {tf.data.experimental.cardinality(test_ds).numpy()}")


Test batches: 1


I0000 00:00:1772680497.321335      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


## Model

In [7]:
# Build model architecture (must match training)
model = keras_cv.models.ImageClassifier.from_preset(
    CFG.preset, num_classes=CFG.num_classes
)

# Compile (needed before load_weights)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.KLDivergence(),
)

# Load trained weights
model.load_weights(str(MODEL_PATH))
print(f"Loaded weights from: {MODEL_PATH.name}")


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 4 variables whereas the saved optimizer has 626 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Loaded weights from: best_model.keras


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 622 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


## Inference

In [8]:
preds = model.predict(test_ds)

# Apply softmax properly (handles negative logits)
preds = preds.astype(np.float64)
preds = np.exp(preds - preds.max(axis=1, keepdims=True))  # numerically stable softmax
preds = preds / preds.sum(axis=1, keepdims=True)

# Safety checks
preds = np.clip(preds, 1e-15, 1.0)
preds = preds / preds.sum(axis=1, keepdims=True)

print(f"Predictions: {preds.shape}")
print(f"Prob sum check: {preds[0].sum():.4f}")
print(f"Min: {preds.min():.6f}, Max: {preds.max():.6f}")
print(f"Any NaN: {np.isnan(preds).any()}")

I0000 00:00:1772680510.454687      78 service.cc:152] XLA service 0x7e367c066c50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772680510.454729      78 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1772680511.892627      78 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1772680521.656568      78 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1/1 ━━━━━━━━━━━━━━━━━━━━ 22s 22s/step
Predictions: (1, 6)
Prob sum check: 1.0000
Min: 0.140821, Max: 0.195205
Any NaN: False


## Create Submission

In [9]:
target_cols = [x.lower() + "_vote" for x in CFG.class_names]
pred_df = test_df[["eeg_id"]].copy()
pred_df[target_cols] = preds.tolist()

sub_df = pd.read_csv(BASE_PATH / "sample_submission.csv")
sub_df = sub_df[["eeg_id"]].copy()
sub_df = sub_df.merge(pred_df, on="eeg_id", how="left")

sub_df.to_csv("submission.csv", index=False)
print(f"submission.csv saved ({len(sub_df)} rows)")
print(sub_df.head())


submission.csv saved (1 rows)
       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  3911565283      0.157292  0.195205  0.140821   0.192601   0.148595   

   other_vote  
0    0.165486  
